# **Day 40: Project - Field Survey Data Manager**

This project closes Phase 2. We will use **functions, files (CSV and GeoJSON), exceptions, dates and logging** together.

## **The Story**
A group of Remote Sensing students went to the **Sundarbans** for a field survey. At each point they recorded:
- a point ID
- the GPS location (latitude, longitude)
- the land cover they saw (mangrove, water, agriculture, settlement, mudflat)
- the date

This data is needed to check the accuracy of a land cover map made from satellite images. But the data was typed by different people, and it has mistakes.

**Our program will:**
1. Read the raw data.
2. **Validate** every row (catch bad values without crashing).
3. Save the good rows to a clean **CSV** file.
4. Export the points as **GeoJSON** so they can be opened in QGIS.
5. Print a **summary** report.

(All data is made up.)

In [1]:
import csv, json, logging, sys
from datetime import datetime
from pathlib import Path

raw = """point_id,lat,lon,land_cover,date,surveyor
SB001,21.9452,88.8931,Mangrove,2024-01-15,Rina
SB002,21.9610,88.9105,Water,2024-01-15,Rina
SB003,21.9877,88.8702,mangrove,2024-01-15,Arko
SB004,22.0150,88.8450,Agriculture,2024-01-16,Arko
SB005,88.8601,21.9500,Mangrove,2024-01-16,Tuhin
SB006,21.9720,88.9020,Mudflat,2024-01-16,Tuhin
SB007,22.0305,88.8211,Settlement,16-01-2024,Rina
SB008,21.9555,,Water,2024-01-17,Arko
SB009,22.0412,88.8390,Agriculture,2024-01-17,Tuhin
SB010,21.9333,88.9200,Forest?,2024-01-17,Rina
SB011,21.9901,88.8801,Mangrove,2024-01-17,Arko
SB012,22.0101,88.8599,Settlement,2024-01-17,Tuhin
"""
Path("sundarbans_raw.csv").write_text(raw, encoding="utf-8")
print("Raw file saved")

Raw file saved


## **Step 1: Set Up Logging**

In [2]:
log = logging.getLogger("survey")
log.setLevel(logging.INFO)
log.handlers.clear()
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter("%(levelname)-7s %(message)s"))
log.addHandler(h)

## **Step 2: A Validation Function**
Each check raises a custom `SurveyError` with a clear message.

The study area box for this survey is:
- Latitude: 21.5 to 22.5
- Longitude: 88.5 to 89.2

In [3]:
class SurveyError(ValueError):
    pass

CLASSES = {"mangrove", "water", "agriculture", "settlement", "mudflat"}
BOX = {"min_lat": 21.5, "max_lat": 22.5, "min_lon": 88.5, "max_lon": 89.2}

def validate(row):
    """Check one row. Return a clean dictionary or raise SurveyError."""
    try:
        lat = float(row["lat"])
        lon = float(row["lon"])
    except ValueError:
        raise SurveyError("latitude or longitude is missing or not a number")

    if not (BOX["min_lat"] <= lat <= BOX["max_lat"] and BOX["min_lon"] <= lon <= BOX["max_lon"]):
        raise SurveyError(f"point ({lat}, {lon}) is outside the study area (swapped?)")

    cover = row["land_cover"].strip().lower()
    if cover not in CLASSES:
        raise SurveyError(f"unknown land cover {row['land_cover']!r}")

    try:
        day = datetime.strptime(row["date"], "%Y-%m-%d").date()
    except ValueError:
        raise SurveyError(f"date {row['date']!r} is not in YYYY-MM-DD format")

    return {"point_id": row["point_id"], "lat": lat, "lon": lon,
            "land_cover": cover.title(), "date": day.isoformat(), "surveyor": row["surveyor"]}

## **Step 3: Read and Validate All Rows**

In [4]:
good, bad = [], []

with open("sundarbans_raw.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        try:
            good.append(validate(row))
        except SurveyError as e:
            bad.append((row["point_id"], str(e)))
            log.warning(f"{row['point_id']}: {e}")

log.info(f"{len(good)} good points, {len(bad)} rejected")

WARNING SB005: point (88.8601, 21.95) is outside the study area (swapped?)


WARNING SB007: date '16-01-2024' is not in YYYY-MM-DD format


WARNING SB008: latitude or longitude is missing or not a number


WARNING SB010: unknown land cover 'Forest?'


INFO    8 good points, 4 rejected


Let us look at why each point was rejected:
- **SB005:** Latitude and longitude were typed in the wrong order.
- **SB007:** The date was written in DD-MM-YYYY format.
- **SB008:** The longitude is missing.
- **SB010:** "Forest?" is not one of our land cover classes.

Also note **SB003**: "mangrove" was written in small letters, but our code fixed it to "Mangrove" instead of rejecting it.

## **Step 4: Save the Clean CSV**

In [5]:
with open("sundarbans_clean.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["point_id", "lat", "lon", "land_cover", "date", "surveyor"])
    writer.writeheader()
    writer.writerows(good)

print(open("sundarbans_clean.csv", encoding="utf-8").read())

point_id,lat,lon,land_cover,date,surveyor
SB001,21.9452,88.8931,Mangrove,2024-01-15,Rina
SB002,21.961,88.9105,Water,2024-01-15,Rina
SB003,21.9877,88.8702,Mangrove,2024-01-15,Arko
SB004,22.015,88.845,Agriculture,2024-01-16,Arko
SB006,21.972,88.902,Mudflat,2024-01-16,Tuhin
SB009,22.0412,88.839,Agriculture,2024-01-17,Tuhin
SB011,21.9901,88.8801,Mangrove,2024-01-17,Arko
SB012,22.0101,88.8599,Settlement,2024-01-17,Tuhin



## **Step 5: Export as GeoJSON**

In [6]:
def to_geojson(points):
    return {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]},
                "properties": {k: v for k, v in p.items() if k not in ("lat", "lon")},
            }
            for p in points
        ],
    }

gj = to_geojson(good)
with open("sundarbans_points.geojson", "w", encoding="utf-8") as f:
    json.dump(gj, f, indent=2)

log.info(f"GeoJSON saved with {len(gj['features'])} points")
print(json.dumps(gj["features"][0], indent=2))

INFO    GeoJSON saved with 8 points


{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      88.8931,
      21.9452
    ]
  },
  "properties": {
    "point_id": "SB001",
    "land_cover": "Mangrove",
    "date": "2024-01-15",
    "surveyor": "Rina"
  }
}


## **Step 6: Summary Report**

In [7]:
from collections import Counter

by_class = Counter(p["land_cover"] for p in good)
by_person = Counter(p["surveyor"] for p in good)
by_day = Counter(p["date"] for p in good)

print("=" * 40)
print("   SUNDARBANS FIELD SURVEY - SUMMARY")
print("=" * 40)
print(f"Total rows      : {len(good) + len(bad)}")
print(f"Accepted        : {len(good)}")
print(f"Rejected        : {len(bad)}")
print("\nPoints per land cover:")
for cls, n in by_class.most_common():
    print(f"   {cls:<12}{n:>3}  {'#' * n}")
print("\nPoints per surveyor:", dict(by_person))
print("Points per day   :", dict(sorted(by_day.items())))

   SUNDARBANS FIELD SURVEY - SUMMARY
Total rows      : 12
Accepted        : 8
Rejected        : 4

Points per land cover:
   Mangrove      3  ###
   Agriculture   2  ##
   Water         1  #
   Mudflat       1  #
   Settlement    1  #

Points per surveyor: {'Rina': 2, 'Arko': 3, 'Tuhin': 3}
Points per day   : {'2024-01-15': 3, '2024-01-16': 2, '2024-01-17': 3}


`Counter` counts things for us automatically. We will study it properly on Day 59.

## **What We Used in This Project**
| Topic | Where |
|---|---|
| Functions | `validate()`, `to_geojson()` |
| Custom exception | `SurveyError` |
| CSV reading and writing | raw and clean files |
| JSON / GeoJSON | map export |
| Dates | checking the date format |
| Logging | warnings for bad rows |

Phase 2 is complete. In Phase 3, we will learn **Object Oriented Programming (OOP)**.